In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dropout, Dense, Activation
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Function to train LSTM model
def train_lstm_model(data):
    chosen_col = 'Close'
    datacol = data.iloc[:, 7:8].values
    sc = MinMaxScaler(feature_range=(0,1))
    scaled_data = sc.fit_transform(datacol)
    X, y = [], []
    time_steps = 50
    for i in range(time_steps, len(data)):
        X.append(scaled_data[i-time_steps:i, 0])
        y.append(scaled_data[i, 0])
    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))
    model = Sequential()
    model.add(LSTM(units=100, input_shape=(X.shape[1], X.shape[2])))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.add(Activation('linear'))
    model.compile(optimizer='adam', loss='mse')
    model.fit(X, y, epochs=50, batch_size=32, verbose=0)
    return model, sc

In [5]:
# Function to train and predict using LSTM model
def train_and_predict_lstm_model(data):
    chosen_col = 'Close'
    datacol = data.iloc[:, 7:8].values
    sc = MinMaxScaler(feature_range=(0,1))
    scaled_data = sc.fit_transform(datacol)
    X, y = [], []
    time_steps = 50
    for i in range(time_steps, len(data)):
        X.append(scaled_data[i-time_steps:i, 0])
        y.append(scaled_data[i, 0])
    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))
    model = Sequential()
    model.add(LSTM(units=100, input_shape=(X.shape[1], X.shape[2])))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.add(Activation('linear'))
    model.compile(optimizer='adam', loss='mse')
    model.fit(X, y, epochs=50, batch_size=32, verbose=0)
    
    # Prediction
    predicted_price = model.predict(X)
    predicted_price = sc.inverse_transform(predicted_price)
    
    return predicted_price

In [13]:
# Function to calculate Sharpe ratio and return portfolio allocation for maximum Sharpe ratio
def calculate_sharpe_ratio(mean_returns, cov_matrix, risk_free_rate=0.0178):
    num_portfolios = 10000
    results = np.zeros((4, num_portfolios))
    all_weights = np.zeros((len(mean_returns), num_portfolios))
    
    for i in range(num_portfolios):
        weights = np.random.random(len(mean_returns))
        weights /= np.sum(weights)
        
        all_weights[:, i] = weights
        
        portfolio_return = np.sum(weights * mean_returns) * 252
        portfolio_stddev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(252)
        
        results[0,i] = portfolio_return
        results[1,i] = portfolio_stddev
        results[2,i] = (portfolio_return - risk_free_rate) / portfolio_stddev
    
    max_sharpe_idx = results[2].argmax()
    max_sharpe_allocation = pd.DataFrame(all_weights[:, max_sharpe_idx], index=Cryptos, columns=['allocation'])
    max_sharpe_allocation['allocation'] = [round(i*100, 2) for i in max_sharpe_allocation['allocation']]
    
    return results, max_sharpe_allocation

In [4]:
Cryptos = ['bitcoin','bitcoin_cash','dash','ethereum','iota','litecoin','monero','nem','neo','numeraire','ripple','stratis','waves']

crypto_data = {}
predicted_prices = {}

crypto_data['bitcoin'] = pd.read_csv('/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Aave.csv', parse_dates=['Date'])
crypto_data['bitcoin_cash'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_BinanceCoin.csv", parse_dates=['Date'])
crypto_data['dash'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Bitcoin.csv", parse_dates=['Date'])
crypto_data['ethereum'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Cardano.csv", parse_dates=['Date'])
crypto_data['iota'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_ChainLink.csv", parse_dates=['Date'])
crypto_data['litecoin'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Cosmos.csv", parse_dates=['Date'])
crypto_data['monero'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_CryptocomCoin.csv", parse_dates=['Date'])
crypto_data['nem'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Dogecoin.csv", parse_dates=['Date'])
crypto_data['neo'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_EOS.csv", parse_dates=['Date'])
crypto_data['numeraire'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Ethereum.csv", parse_dates=['Date'])
crypto_data['ripple'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Iota.csv", parse_dates=['Date'])
crypto_data['stratis'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Litecoin.csv", parse_dates=['Date'])
crypto_data['waves'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Monero.csv", parse_dates=['Date'])
crypto_data['ethereum'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_NEM.csv", parse_dates=['Date'])
crypto_data['iota'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Polkadot.csv", parse_dates=['Date'])
crypto_data['litecoin'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Solana.csv", parse_dates=['Date'])
crypto_data['monero'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Stellar.csv", parse_dates=['Date'])
crypto_data['nem'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Tether.csv", parse_dates=['Date'])
crypto_data['neo'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Tron.csv", parse_dates=['Date'])
crypto_data['numeraire'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Uniswap.csv", parse_dates=['Date'])
crypto_data['ripple'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_USDCoin.csv", parse_dates=['Date'])
crypto_data['stratis'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_WrappedBitcoin.csv", parse_dates=['Date'])
crypto_data['waves'] = pd.read_csv("/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_XRP.csv", parse_dates=['Date'])

In [7]:
for crypto in Cryptos:
    crypto_data[crypto] = pd.read_csv(f'/Users/Phoestia/Desktop/StoneLake/Algorithm_Trading/Coin_Price/coin_Bitcoin.csv')
    predicted_price = train_and_predict_lstm_model(crypto_data[crypto])
    predicted_prices[crypto] = predicted_price.flatten()

2023-09-10 00:33:04.278271: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [8]:
# Create DataFrame for predicted future prices
df_predicted = pd.DataFrame(predicted_prices)

In [9]:
# Calculate expected returns and covariance matrix based on predicted prices
returns = df_predicted.pct_change()
mean_returns = returns.mean()
cov_matrix = returns.cov()

In [14]:
# Optimize portfolio to maximize Sharpe ratio
results, max_sharpe_allocation = calculate_sharpe_ratio(mean_returns, cov_matrix)

In [11]:
# Extract the portfolio with maximum Sharpe ratio
return_sharpe_max = results[0,results[2].argmax()]
risk_sharpe_max = results[1,results[2].argmax()]

In [15]:
print(f"Maximum Sharpe Ratio Portfolio Return: {return_sharpe_max}")
print(f"Maximum Sharpe Ratio Portfolio Risk: {risk_sharpe_max}")
print("\nPortfolio Allocation for Maximum Sharpe Ratio:")
print(max_sharpe_allocation)

Maximum Sharpe Ratio Portfolio Return: 0.5367596107270775
Maximum Sharpe Ratio Portfolio Risk: 0.4164596770860822

Portfolio Allocation for Maximum Sharpe Ratio:
              allocation
bitcoin             0.75
bitcoin_cash        1.14
dash                6.91
ethereum           11.50
iota                2.31
litecoin            6.46
monero             12.00
nem                12.85
neo                 7.93
numeraire          14.45
ripple              0.00
stratis            17.42
waves               6.27
